**Workshop notebooks:** [01 — Matching & Loading](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/01_matching_and_loading.ipynb)  &nbsp;·&nbsp; **02 — Overlay & Enrichment(you are here)** &nbsp;·&nbsp;[03 — Bridging](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/03_bridging.ipynb)&nbsp;·&nbsp;[04 — Disease Modules (optional)](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/04_disease_modules_optional.ipynb)

# 02 — Overlay & Enrichment

**Network Medicine Workshop · Kidney Disease · Part 2 of 3**

Now that everything is matched and loaded, we bring each dataset together with its network:

1. **Overlay** — map transcripts onto the transcriptome network, proteins onto the PPI network, metabolites onto the metabolite network, and check coverage
2. **Extract the connected module** — within each network, do your DE nodes form a connected piece, or are they scattered?
3. **Enrichment** — what biology does that module represent (GO terms, KEGG/Reactome pathways)?

Requires that you've already run Notebook 1 (or have its `processed/` outputs available).

*Curious whether that connectivity is statistically more than you'd expect by chance, or how your module compares to other diseases? That's the optional Notebook 4 — kept separate since it's an extension.*


**New here?** Run cells top to bottom with Shift+Enter (or the ▶ button), and wait for each one to finish before running the next — later cells depend on variables set by earlier ones. See Notebook 1 for a fuller intro to Colab if this is your first time.

## Setup — reload everything from Notebook 1

This notebook builds directly on Notebook 1's output, so the first few cells reinstall the small set of packages this notebook needs, reconnect to your Drive, and load back the matched tables and network graphs that were saved to `processed/` last time. If you're running this in the same Colab session as Notebook 1 it'll feel instant. In a fresh session (or on a different day) it takes a few seconds to reload everything from Drive.

In [ ]:
!pip install -q gprofiler-official networkx matplotlib


Drive needs to be reconnected in every new Colab session, even though the files themselves are still there. The cell below does that; click through the permission prompt just like in Notebook 1.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


With Drive connected, this next cell reloads everything Notebook 1 produced: the three network graphs (saved as `pickle` files, which preserve the exact graph object) and the three matched tables (saved as CSVs).

In [ ]:
import os, sys, pickle
import pandas as pd
import networkx as nx

BASE_DIR = "/content/drive/MyDrive/network_medicine_workshop"
PROC_DIR = os.path.join(BASE_DIR, "processed")

sys.path.insert(0, os.path.join(BASE_DIR, "helpers"))
_helper_file = os.path.join(BASE_DIR, "helpers", "nb2_helpers.py")
if not os.path.exists(_helper_file):
    # Rare, but can happen if Drive's sync lags behind a fresh download, or this notebook runs
    # before Notebook 1 has -- fetch this one helper file directly instead of leaving you stuck.
    import requests, importlib
    r = requests.get(f"https://raw.githubusercontent.com/marlene-grabner/NetworkMedicine_Workshop/main/helpers/nb2_helpers.py", timeout=30)
    r.raise_for_status()
    with open(_helper_file, "wb") as f:
        f.write(r.content)
    importlib.invalidate_caches()
    print(f"[recovered] fetched missing nb2_helpers.py directly")

from nb2_helpers import plot_module_overview   # figure code — see helpers/nb2_helpers.py

LAYERS = ["transcriptome", "ppi", "metabolite"]

graphs = {}
for layer in LAYERS:
    with open(os.path.join(PROC_DIR, f"graph_{layer}.pkl"), "rb") as f:
        graphs[layer] = pickle.load(f)

degs_matched = pd.read_csv(os.path.join(PROC_DIR, "degs_matched.csv"), dtype={"ncbi_gene_id": str})
proteins_matched = pd.read_csv(os.path.join(PROC_DIR, "proteins_matched.csv"), dtype={"ncbi_gene_id": str})
metabs_matched = pd.read_csv(os.path.join(PROC_DIR, "metabolites_matched.csv"), dtype={"kegg_id": str})

print("Reloaded 3 networks and 3 matched tables from", PROC_DIR)


Each network layer is fed by its own DE dataset — transcripts feed the transcriptome network, proteins feed the PPI network, metabolites feed the metabolite network. The next cell collects each dataset's matched IDs into a simple `layer -> set of IDs` lookup, `LAYER_DE_IDS`, that the rest of the notebook uses to know which network nodes are your differentially expressed ones.

In [ ]:
# each network layer has its own DE dataset feeding it
LAYER_DE_IDS = {
    "transcriptome": set(degs_matched["ncbi_gene_id"].dropna().unique()),
    "ppi":           set(proteins_matched["ncbi_gene_id"].dropna().unique()),
    "metabolite":    set(metabs_matched["kegg_id"].dropna().unique()),
}

for layer in LAYERS:
    print(f"{layer}: {len(LAYER_DE_IDS[layer])} matched DE entities, "
          f"network has {graphs[layer].number_of_nodes():,} nodes / {graphs[layer].number_of_edges():,} edges")


## Step 1 — Overlay: how much of your DE list actually lands on the network?

Not every DE entity will be a node in its network (coverage gaps are normal — networks are incomplete, and DE hits may not be represented in the interactome/co-expression net/metabolite net at all). This coverage number matters for everything downstream: low coverage means a weaker signal to work with.


In [ ]:
module_seed_nodes = {}   # DE-mapped nodes that are actually present in each network
for layer, G in graphs.items():
    de_set = LAYER_DE_IDS[layer]
    present = de_set & set(G.nodes())
    module_seed_nodes[layer] = present
    print(f"[{layer}] {len(present)} / {len(de_set)} DE entities present as network nodes "
          f"({len(present)/max(len(de_set),1):.1%} coverage)")


## Step 2 — Extract the connected DE module

Within each network, look only at the subgraph induced by your DE nodes: do they form a connected piece (a "module"), or are they scattered across disconnected single nodes? We report the largest connected component per layer as a descriptive statistic — this is the set of DE nodes that are directly or indirectly wired to each other and forms the basis for the cross-omics bridging in Notebook 3.


We use two different notions of "connected" here: the single **largest** connected component (a quick headline number), and the union of **all** components with more than one node (a fuller picture of everything that's wired together, not just the single biggest cluster). The two functions below compute each, and we'll use both.

In [ ]:
def largest_cc_nodes(G, nodes):
    if len(nodes) < 2:
        return set(nodes)
    sub = G.subgraph(nodes)
    if sub.number_of_edges() == 0:
        return set()
    return max(nx.connected_components(sub), key=len)

def connected_module_nodes(G, seed_nodes):
    """Union of all connected components (size > 1) among the DE-mapped nodes."""
    if len(seed_nodes) < 2:
        return set()
    sub = G.subgraph(seed_nodes)
    connected = set()
    for comp in nx.connected_components(sub):
        if len(comp) > 1:
            connected |= comp
    return connected


Now we apply both functions to every network layer and print the results — this module-extraction step is the one the rest of the workshop builds on.

In [ ]:
module_connected_nodes = {}
module_lcc_nodes = {}
for layer, G in graphs.items():
    seed_nodes = module_seed_nodes[layer]
    lcc = largest_cc_nodes(G, seed_nodes)
    connected = connected_module_nodes(G, seed_nodes)
    module_connected_nodes[layer] = connected
    module_lcc_nodes[layer] = lcc
    n_components = nx.number_connected_components(G.subgraph(seed_nodes)) if len(seed_nodes) > 1 else 0
    print(f"[{layer}] largest connected component: {len(lcc)} nodes  |  "
          f"total nodes in any connected piece (size>1): {len(connected)} / {len(seed_nodes)}  |  "
          f"components: {n_components}")


## A quick look at each module

All three modules side by side — the largest connected component in each network, plus a bit of the surrounding network for context.

Each panel deliberately does **not** use a plain force-directed layout on the combined graph — that pulls well-connected context nodes right into the middle, which visually (and wrongly) suggests they're just as "central" as the module itself. Instead, the module is laid out as a tight cluster in the middle, and every context node is placed out near whichever module node it actually attaches to — so distance from the center genuinely means "how connected to the module," making the module's localization obvious at a glance. That layout logic (plus the rest of the plotting code) lives in `helpers/nb2_helpers.py` if you want to see exactly how the figure below is built — it's plain Python, nothing hidden, just moved out of the way so this notebook can stay focused on the network-medicine part. Singleton DE nodes (not connected to anything) are left out, same as before.

In [ ]:
plot_module_overview(graphs, module_seed_nodes, module_lcc_nodes, LAYERS, PROC_DIR)


## Step 3 — Enrichment: what biology is this?

We run functional enrichment (GO terms, KEGG/Reactome pathways) via [g:Profiler](https://biit.cs.ut.ee/gprofiler/). We do this twice per gene-based layer (transcriptome, PPI):

- on the **raw DE list**, and
- on the **connected module only** (the DE nodes that formed a connected component above)

Comparing the two is informative: the module-only enrichment is usually cleaner/more specific, since it's filtered to genes that are network-validated as belonging together, not just individually differentially expressed.


The helper below wraps a single g:Profiler call: it skips layers with too few genes (results get unstable and hard to interpret below ~3), and otherwise prints and displays the most significant terms.

In [ ]:
from gprofiler import GProfiler

gp = GProfiler(return_dataframe=True)

def run_enrichment(gene_ids, label, sources=("GO:BP", "KEGG", "REAC")):
    if len(gene_ids) < 3:
        print(f"[{label}] too few genes ({len(gene_ids)}) — skipping")
        return None
    res = gp.profile(organism="hsapiens", query=list(gene_ids), sources=list(sources))
    if res.empty:
        print(f"[{label}] no significant terms")
        return res
    res = res.sort_values("p_value")
    print(f"[{label}] top terms:")
    display(res[["source", "name", "p_value", "term_size", "intersection_size"]].head(10))
    return res


Now we run it for each gene-based layer (transcriptome, PPI), once on the full DE list and once on the connected-module-only subset — four calls in total. Comparing the two per layer is the interesting part: the module-only result is usually cleaner, since it's filtered to genes that are network-validated as belonging together, not just individually differentially expressed.

In [ ]:
GENE_LAYERS = ["transcriptome", "ppi"]

enrichment_results = {}
for layer in GENE_LAYERS:
    seed_nodes = module_seed_nodes[layer]
    connected = module_connected_nodes[layer]
    enrichment_results[f"{layer}_full_de_list"] = run_enrichment(seed_nodes, f"{layer} — full DE list")
    enrichment_results[f"{layer}_connected_module"] = run_enrichment(connected, f"{layer} — connected module only")


## Step 4 — Save outputs for later notebooks

Last step: write the seed/connected node sets for every layer, plus every enrichment table we computed, to `processed/`. Notebook 3 picks the protein module back up from here to bridge it against the metabolite module.

In [ ]:
modules = {
    layer: {"seed_nodes": module_seed_nodes[layer], "connected_nodes": module_connected_nodes[layer]}
    for layer in LAYERS
}

with open(os.path.join(PROC_DIR, "modules.pkl"), "wb") as f:
    pickle.dump(modules, f)

for name, res in enrichment_results.items():
    if res is not None and not res.empty:
        res.to_csv(os.path.join(PROC_DIR, f"enrichment_{name}.csv"), index=False)

print("Saved modules and enrichment tables to", PROC_DIR)


---
**Next:** open `03_bridging.ipynb` to connect the protein module and the metabolite module, and see whether they converge on the same biology — and whether any of it looks targetable.

**Optional:** `04_disease_modules_optional.ipynb` picks up the module-significance question we skipped here, and extends it to compare your module against two reference diseases.


**Workshop notebooks:** [01 — Matching & Loading](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/01_matching_and_loading.ipynb)  &nbsp;·&nbsp; **02 — Overlay & Enrichment(you are here)** &nbsp;·&nbsp;[03 — Bridging](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/03_bridging.ipynb)&nbsp;·&nbsp;[04 — Disease Modules (optional)](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/04_disease_modules_optional.ipynb)